### Widoki analityczne

- Przykład 1 – liczba odwołanych lotów per lotnisko z nazwą miasta

In [0]:
%sql
CREATE OR REPLACE VIEW flight_sales_gold.vw_airport_cancelled AS
SELECT
    f.date_key,
    ap.airport_name,
    ap.city,
    SUM(f.cancelled) AS num_cancelled
FROM flight_sales_silver.dim_route as r 
JOIN flight_sales_silver.dim_airports ap
    ON r.origin_airport_key = ap.airport_key
JOIN flight_sales_gold.fact_flights f
    ON r.route_key = f.route_key
GROUP BY f.date_key, ap.airport_name, ap.city
ORDER BY num_cancelled DESC;

In [0]:
%sql 
SELECT * FROM flight_sales_gold.vw_airport_cancelled


date_key,airport_name,city,num_cancelled
2015-12-28,Chicago O'Hare International Airport,Chicago,1210
2015-02-23,Dallas/Fort Worth International Airport,Dallas-Fort Worth,1022
2015-02-28,Dallas/Fort Worth International Airport,Dallas-Fort Worth,936
2015-06-15,Chicago O'Hare International Airport,Chicago,778
2015-05-10,Dallas/Fort Worth International Airport,Dallas-Fort Worth,732
2015-03-05,Dallas/Fort Worth International Airport,Dallas-Fort Worth,724
2015-04-09,Chicago O'Hare International Airport,Chicago,712
2015-02-27,Dallas/Fort Worth International Airport,Dallas-Fort Worth,710
2015-11-21,Chicago O'Hare International Airport,Chicago,632
2015-12-27,Dallas/Fort Worth International Airport,Dallas-Fort Worth,574


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

- Przykład 2 - liczba lotów per data z nazwą linii lotniczej 

In [0]:
%sql
CREATE OR REPLACE VIEW flight_sales_gold.vw_airline_daily_delay AS
SELECT
    f.date_key,
    a.airline_name,
    AVG(f.arrival_delay) AS avg_arrival_delay,
    AVG(f.departure_delay) AS avg_departure_delay,
    COUNT(*) AS num_flights
FROM flight_sales_gold.fact_flights f
JOIN flight_sales_silver.dim_airlines a
    ON f.airline_key = a.airline_key
GROUP BY f.date_key, a.airline_name;


In [0]:
%sql 
SELECT * FROM flight_sales_gold.vw_airline_daily_delay

date_key,airline_name,avg_arrival_delay,avg_departure_delay,num_flights
2015-12-28,Virgin America,8.473958333333334,16.302083333333332,388
2015-10-08,Atlantic Southeast Airlines,0.24229346485819975,3.127921279212792,3254
2015-11-04,Alaska Airlines Inc.,-2.995661605206074,2.002164502164502,924
2015-10-12,American Airlines Inc.,-7.35678585159554,1.7991551459293396,5216
2015-11-10,Alaska Airlines Inc.,-3.958515283842795,-1.2882096069868996,916
2015-08-20,Spirit Air Lines,33.93769470404985,33.417956656346746,674
2015-11-15,Alaska Airlines Inc.,3.8389830508474576,6.680761099365751,946
2015-08-21,Spirit Air Lines,35.13636363636363,38.616314199395774,682
2015-08-28,American Eagle Airlines Inc.,-3.3138500635324015,4.0444162436548226,1582
2015-09-04,Alaska Airlines Inc.,9.425742574257425,8.577075098814229,1012


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

- Przykład 3 - ile danego dnia odwołano lotów z powodu różnych przyczyn 

In [0]:
%sql
CREATE OR REPLACE VIEW flight_sales_gold.vw_cancellation_reason AS
SELECT 
    f.date_key,
    cr.code,
    cr.description,
    SUM(f.cancelled) AS num_cancelled
FROM flight_sales_gold.fact_flights f
JOIN flight_sales_silver.dim_cancellation_reason AS cr 
    ON f.cancellation_reason_key = cr.cancellation_reason_key
GROUP BY f.date_key, cr.code, cr.description;


In [0]:
%sql 
SELECT * FROM flight_sales_gold.vw_cancellation_reason 

date_key,code,description,num_cancelled
2015-05-19,B,Weather,26
2015-02-20,C,National Air System,28
2015-05-30,C,National Air System,21
2015-02-10,A,Carrier,65
2015-03-14,A,Carrier,36
2015-06-05,C,National Air System,13
2015-06-01,C,National Air System,226
2015-04-08,B,Weather,32
2015-03-24,A,Carrier,110
2015-01-18,B,Weather,39


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

### Dobowa granulacja 
- grupuje fakty po `date_key` i liczę miary 

In [0]:
%sql
-- Agregacja dobowa: średnie opóźnienia per dzień
SELECT
    date_key,
    AVG(arrival_delay) AS avg_arrival_delay,
    AVG(departure_delay) AS avg_departure_delay
FROM flight_sales_gold.fact_flights
GROUP BY date_key
ORDER BY date_key;


date_key,avg_arrival_delay,avg_departure_delay
2015-01-01,5.352495543672014,9.610896960711639
2015-01-02,9.838903743315509,12.649745269286754
2015-01-03,25.461860465116278,25.168418964491174
2015-01-04,31.975011015295525,31.567859384808536
2015-01-05,18.81131019036954,21.116838062197992
2015-01-06,21.299273998386663,22.486405036163944
2015-01-07,11.955428646448732,14.522800130847235
2015-01-08,13.316482498511215,16.40372752329702
2015-01-09,12.25561145510836,15.368481964894233
2015-01-10,1.9224748723018619,8.200148014143574
